# Stage 00 – Data Preprocess and Merge
Merge current season Quotazioni with historical stats and compute Age from DOB.
Manual updates needed only for Regularness and Mate after this stage.

In [1]:
import os, json
import pandas as pd
from pathlib import Path

# Detect repo root when notebook is opened from pipeline/
BASE = os.getcwd()
if os.path.basename(BASE) == 'pipeline':
    BASE = os.path.abspath(os.path.join(BASE, '..'))
os.chdir(BASE)

SEASON = "25-26"
SEASON_START_YEAR = 2025
RAW_DIR = f"data/raw/{SEASON}"
RAW_HIST_DIR = "data/raw/historical"
INTER_DIR = f"data/intermediate/{SEASON}"
HIST_DIR = "data/historical"
UTILS_DIR = "data/utils"

QUOTAZIONI_PATH = f"{RAW_DIR}/Quotazioni_Fantacalcio_Stagione_2025_26.xlsx"
DOB_PATH = f"{UTILS_DIR}/player_dob.csv"
OUTPUT_PATH = f"{INTER_DIR}/data_preprocess_merge.xlsx"
os.makedirs(INTER_DIR, exist_ok=True)


## 0. Setup constants

In [2]:
fh_to_fc = {
    'Id':'Id','Role':'R','Role_M':'RM','Name':'Nome','Squad':'Squadra','Price':'Qt.A','FVM':'FVM','Pg':'Pv','Mv':'Mv','Mf':'Fm'
}
unneeded_labels_new_df = ['Qt.I','Qt.I M','Qt.A M','Diff.','Diff.M','FVM M']
unneeded_labels_stats = ['Amm','Ass','Au','Esp','Gf','Gs','Rp','Rc','R+','R-','Rm']

## 1. Load inputs

In [3]:
df_quotazioni = pd.read_excel(QUOTAZIONI_PATH, header=1)
df_quotazioni = df_quotazioni.rename(columns={v:k for k,v in fh_to_fc.items()})
df_quotazioni = df_quotazioni.drop(columns=[c for c in unneeded_labels_new_df if c in df_quotazioni.columns], errors="ignore")
df_dob = pd.read_csv(DOB_PATH)
df_dob['DOB'] = pd.to_datetime(df_dob['DOB'], errors='coerce')

stats_files = sorted([f for f in os.listdir(HIST_DIR) if f.endswith('.xlsx')], reverse=True)
stats_dfs = {}
for f in stats_files:
    df = pd.read_excel(os.path.join(HIST_DIR, f), header=1)
    df = df.drop(columns=[c for c in unneeded_labels_stats if c in df.columns], errors='ignore')
    stats_dfs[os.path.splitext(f)[0]] = df

hist_quot_files = sorted([f for f in os.listdir(RAW_HIST_DIR) if f.endswith('.xlsx')], reverse=True)
hist_quot_dfs = {}
for f in hist_quot_files:
    df = pd.read_excel(os.path.join(RAW_HIST_DIR, f), header=1)
    df = df.rename(columns={v:k for k,v in fh_to_fc.items()})
    df = df[['Id','FVM']].dropna(subset=['Id','FVM'])
    hist_quot_dfs[os.path.splitext(f)[0]] = df

## 2. Compute Age from DOB

In [4]:
def compute_age(dob):
    if pd.isna(dob):
        return pd.NA
    age = SEASON_START_YEAR - dob.year
    if (dob.month, dob.day) > (8, 31):
        age -= 1
    return int(age)

df_quotazioni = df_quotazioni.merge(df_dob[['Id','DOB']], on='Id', how='left')
df_quotazioni['Age'] = df_quotazioni['DOB'].apply(compute_age)

# Export missing DOBs for manual/LLM fill
missing_dob = df_quotazioni[df_quotazioni['DOB'].isna()][['Id','Name','Squad']]
if not missing_dob.empty:
    missing_path = f"{INTER_DIR}/missing_dob_{SEASON}.csv"
    missing_dob.to_csv(missing_path, index=False)
    raise SystemExit(f"Missing DOB for {len(missing_dob)} players → {missing_path}. Fill data/utils/player_dob.csv and re-run.")


## 3. Merge historical stats

In [5]:
cols = ['Id','Role','Role_M','Name','Squad','Price','Age','FVM']
merged = df_quotazioni[cols].copy()
# Manual-intervention columns (by design filled by hand after this stage, before Stage 1):
# Mate = suggested teammate (player name), Regularness = regularity/injury score.
merged.insert(list(merged.columns).index('FVM'), 'Mate', '')
merged.insert(list(merged.columns).index('FVM'), 'Regularness', float('nan'))
for name, df in stats_dfs.items():
    season = name[-5:]
    tmp = df[['Id', fh_to_fc['Pg'], fh_to_fc['Mv'], fh_to_fc['Mf']]].rename(columns={fh_to_fc['Pg']:f'Pg{season}', fh_to_fc['Mv']:f'Mv{season}', fh_to_fc['Mf']:f'Mf{season}'} )
    merged = merged.merge(tmp, on='Id', how='left')
for name, df in hist_quot_dfs.items():
    season = name[-5:]
    # Only keep FVM for seasons where stats exist, i.e. 22_23 onwards
    tmp = df[['Id','FVM']].rename(columns={'FVM':f'FVM{season}'})
    merged = merged.merge(tmp, on='Id', how='left')
new_df = merged

## 4. Output & Validation

In [6]:
new_df.to_excel(OUTPUT_PATH, index=False)
report = {
    "season": SEASON,
    "rows_output": int(len(new_df)),
    "age_missing": int(new_df['Age'].isna().sum()),
    "id_unique": bool(new_df['Id'].nunique() == len(new_df))
}
with open(f"{INTER_DIR}/validation_stage0.json", "w") as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

{
  "season": "25-26",
  "rows_output": 532,
  "age_missing": 0,
  "id_unique": true
}
